# View Database Tables
This notebook connects to the PostgreSQL database and displays the contents of the products and reviews tables, using the structure from `load_dataframes.nbconvert.ipynb`.

In [1]:
import psycopg2
import pandas as pd
import sys
import json
import ast
import re
import os
import csv
from io import StringIO


In [2]:
# --- Database Connection Details ---
DB_NAME = "amazon_electronics_rag"
DB_USER = "akshat"
DB_PASS = "2139"
DB_HOST = "localhost"
DB_PORT = "5432"

In [3]:
def load_data():
    """
    Connects to the PostgreSQL database and loads products_laptop and reviews_laptop
    tables into pandas DataFrames.
    """
    conn = None
    try:
        print(f"Connecting to database '{DB_NAME}' on {DB_HOST}...")
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASS,
            host=DB_HOST,
            port=DB_PORT
        )
        print("Connection successful.")
  

        # Load reviews_laptop
        print("\nLoading 'reviews_laptop' table...")

        
        query_reviews = "select review_id,parent_asin,user_id,title,verified_purchase,text from reviews_laptop WHERE parent_asin IN (SELECT parent_asin FROM products_laptop WHERE price IS NOT NULL AND NOT 'Laptop Travel Accessories' = ANY(categories));"
        df_reviews = pd.read_sql_query(query_reviews, conn)
        print(f"Loaded {len(df_reviews)} rows from 'reviews_laptop'.")

        
        
        
        
        return df_reviews

    except (Exception, psycopg2.Error) as error:
        print(f"Error: {error}", file=sys.stderr)
        return None, None

    finally:
        if conn:
            conn.close()
            print("\nPostgreSQL connection closed.")

In [4]:
df_reviews = load_data()

Connecting to database 'amazon_electronics_rag' on localhost...
Connection successful.

Loading 'reviews_laptop' table...


/tmp/ipykernel_21366/2548910740.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reviews = pd.read_sql_query(query_reviews, conn)


Loaded 178187 rows from 'reviews_laptop'.

PostgreSQL connection closed.


In [5]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178187 entries, 0 to 178186
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   review_id          178187 non-null  int64 
 1   parent_asin        178187 non-null  object
 2   user_id            178187 non-null  object
 3   title              178187 non-null  object
 4   verified_purchase  178187 non-null  bool  
 5   text               178187 non-null  object
dtypes: bool(1), int64(1), object(4)
memory usage: 7.0+ MB


In [6]:
folder_path = '/home/akshat/CSE_573/laptop_hybrid_search/agentic_hybrid_search/graph_db/python_scripts/data'

# Define the filename
file_name = 'reviews.csv'

# Combine the folder path and filename to create the full file path
full_file_path = os.path.join(folder_path, file_name)

# Ensure the directory exists (create it if it doesn't)
os.makedirs(folder_path, exist_ok=True)

# Save the DataFrame to the specified CSV file
df_reviews[["review_id","parent_asin","verified_purchase","user_id"]].to_csv(full_file_path, index=False) # index=False prevents writing the DataFrame index as a column

In [16]:
df_text_sent = pd.read_csv("/home/akshat/CSE_573/laptop_hybrid_search/agentic_hybrid_search/graph_db/python_scripts/review_csvs/reviews_parsed_final .csv")

In [17]:
df_text_sent.head()

,review_id,parsed_data
0,8040,"{\n ""Battery"": null,\n ""Screen"": {""sentiment..."
1,8043,"{\n ""Battery"": null,\n ""Screen"": null,\n ""P..."
2,8048,"{\n ""Battery"": null,\n ""Screen"": null,\n ""P..."
3,8178,"{\n ""Battery"": null,\n ""Screen"": null,\n ""P..."
4,8050,"{\n ""Battery"": {""sentiment"": ""Neutral"", ""reas..."


In [18]:
def flatten_parsed_data(row):
    try:
        data = json.loads(row['parsed_data'])
    except (json.JSONDecodeError, TypeError):
        return []

    extracted_rows = []
    target_features = ['Battery', 'Screen', 'Performance']
    
    for feature in target_features:
        details = data.get(feature)
        
        if details and isinstance(details, dict):
            sentiment = details.get('sentiment')
            reasons = details.get('reasons', [])
            
            reasons_str = "; ".join(reasons) if isinstance(reasons, list) else str(reasons)
            if sentiment:
                extracted_rows.append({
                    'review_id': row['review_id'],
                    'feature': feature,         # This will be your Aspect Node
                    'sentiment': sentiment,     # This will be the Relationship Type/Property
                    'reasons': reasons_str      # This will be a Property on the Relationship
                })
                
    return extracted_rows

# 3. Apply the function and create the new DataFrame
# We use a list comprehension for speed
structured_data = [
    item 
    for _, row in df_text_sent.iterrows() 
    for item in flatten_parsed_data(row)
]

df_structured = pd.DataFrame(structured_data)

# 4. Check the results
print(f"Original Rows: {len(df_text_sent)}")
print(f"Structured Rows: {len(df_structured)}")
print("\nSample Data:")


# Optional: Save to CSV
# df_structured.to_csv("reviews_structured_for_graph.csv", index=False)

Original Rows: 178187
Structured Rows: 232153

Sample Data:
   review_id      feature sentiment  \
0       8040       Screen  Negative   
1       8043  Performance  Negative   
2       8048  Performance  Negative   
3       8178  Performance  Negative   
4       8050      Battery   Neutral   

                                             reasons  
0  poor color quality; blacks are washed out; can...  
1  BSOD; reboots frequently; redundant reboots fo...  
2  have to click 2 or 3 times for it to work; can...  
3  advertising discrepancy; receiving lower perfo...  
4                                      not mentioned  


In [20]:
df_structured.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232153 entries, 0 to 232152
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   review_id  232153 non-null  int64 
 1   feature    232153 non-null  object
 2   sentiment  232153 non-null  object
 3   reasons    232153 non-null  object
dtypes: int64(1), object(3)
memory usage: 7.1+ MB


In [21]:
folder_path = '/home/akshat/CSE_573/laptop_hybrid_search/agentic_hybrid_search/graph_db/python_scripts/data'

# Define the filename
file_name = 'reviews_sentiment.csv'

# Combine the folder path and filename to create the full file path
full_file_path = os.path.join(folder_path, file_name)

# Ensure the directory exists (create it if it doesn't)
os.makedirs(folder_path, exist_ok=True)

# Save the DataFrame to the specified CSV file
df_structured.to_csv(full_file_path, index=False) # index=False prevents writing the DataFrame index as a column